<a href="https://colab.research.google.com/github/Kibet-Rotich/DeepLearning/blob/master/GraphNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q torch-geometric pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

# ==========================================
# 1. SETUP GRAPH DATA
# ==========================================
# 4 nodes, each with 3 input features
x = torch.tensor([
    [1.0, 0.0, 0.5],  # Node 0
    [0.0, 1.0, 1.5],  # Node 1
    [1.0, 1.0, 0.0],  # Node 2
    [0.2, 0.8, 1.1]   # Node 3
], dtype=torch.float)

# Directed/Undirected edges in COO format [2, num_edges]
# Connections: 0-1, 1-0, 1-2, 2-1, 2-3, 3-2
edge_index = torch.tensor([
    [0, 1, 1, 2, 2, 3],
    [1, 0, 2, 1, 3, 2]
], dtype=torch.long)

# Binary labels for each node (e.g., Class 0 or Class 1)
y = torch.tensor([0, 1, 1, 0], dtype=torch.long)

# Train/Test boolean masks
train_mask = torch.tensor([True, True, True, False])
test_mask = torch.tensor([False, False, False, True])

# Package into a PyG Data object
data = Data(x=x, edge_index=edge_index, y=y, train_mask=train_mask, test_mask=test_mask)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data = data.to(device)


# ==========================================
# 2. DEFINE THE GNN ARCHITECTURE
# ==========================================
class GCN(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int):
        super().__init__()
        # GCNConv automatically handles neighbor aggregation & weight transformation
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        # Layer 1: Aggregate 1-hop neighbor features
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.2, training=self.training)

        # Layer 2: Aggregate 2-hop neighbor features
        x = self.conv2(x, edge_index)
        return x  # Raw logits for each node: [num_nodes, out_channels]


model = GCN(in_channels=3, hidden_channels=8, out_channels=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
loss_fn = nn.CrossEntropyLoss()


# ==========================================
# 3. TRAINING LOOP (Transductive Node Classification)
# ==========================================
model.train()
for epoch in range(1, 51):
    optimizer.zero_grad()

    # Forward pass uses BOTH node features AND the graph topology
    out = model(data.x, data.edge_index)

    # Compute loss ONLY on the training nodes using the mask
    loss = loss_fn(out[data.train_mask], data.y[data.train_mask])

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch [{epoch:02d}/50] - Loss: {loss.item():.4f}")


# ==========================================
# 4. EVALUATION & INFERENCE
# ==========================================
model.eval()
with torch.no_grad():
    logits = model(data.x, data.edge_index)
    predictions = logits.argmax(dim=-1)

# Check predictions on the unseen test node (Node 3)
test_pred = predictions[data.test_mask]
test_true = data.y[data.test_mask]

print(f"\nPredicted classes for all nodes: {predictions.tolist()}")
print(f"Test Node 3 -> Pred: {test_pred.item()}, True: {test_true.item()}")

Epoch [10/50] - Loss: 0.6237
Epoch [20/50] - Loss: 0.5742
Epoch [30/50] - Loss: 0.5709
Epoch [40/50] - Loss: 0.6039
Epoch [50/50] - Loss: 0.5966

Predicted classes for all nodes: [1, 1, 1, 1]
Test Node 3 -> Pred: 1, True: 0


# GNN Recommender System (Link Prediction)
In this notebook, we build a heterogeneous Graph Neural Network to recommend movies to users.
Instead of predicting node classes, we predict whether an **edge** (a rating/interaction) should exist between a User node and a Movie node.

## 1. Imports and Device Setup
We import standard PyTorch libraries alongside PyG specific modules like `SAGEConv` (GraphSAGE) and `to_hetero` (which automatically converts standard homogeneous GNNs to handle heterogeneous graphs).

In [3]:
import torch
import torch.nn.functional as F
import torch_geometric.transforms as T
from torch_geometric.datasets import MovieLens
from torch_geometric.nn import SAGEConv, to_hetero
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 2. Loading the Heterogeneous Graph
We download the MovieLens-100k dataset directly via PyG.
Because information must flow both ways (User $\rightarrow$ Movie, and Movie $\rightarrow$ User), we use `T.ToUndirected()` to add reverse edges.

In [4]:
# Download and format the dataset
dataset = MovieLens(root='./data/ml-100k', model_name='all-MiniLM-L6-v2')
data = dataset[0]

# Add reverse edges so message passing flows in both directions
data = T.ToUndirected()(data)

print("Heterogeneous Graph Structure:")
print(data)

print("\nNotice the node types ('user' and 'movie') and the edge types ('rates' and 'rev_rates').")

Extracting data/ml-100k/raw/ml-latest-small.zip
Processing...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/305 [00:00<?, ?it/s]

Heterogeneous Graph Structure:
HeteroData(
  movie={ x=[9742, 404] },
  user={ num_nodes=610 },
  (user, rates, movie)={
    edge_index=[2, 100836],
    edge_label=[100836],
    time=[100836],
  },
  (movie, rev_rates, user)={
    edge_index=[2, 100836],
    edge_label=[100836],
    time=[100836],
  }
)

Notice the node types ('user' and 'movie') and the edge types ('rates' and 'rev_rates').


Done!


## 3. Data Splitting for Recommendation
In node classification, we hide node labels. In link prediction, we hide **edges**.
`RandomLinkSplit` removes a percentage of the `user -> rates -> movie` edges to use as our validation and test sets.
It also generates **negative samples** (movies the user has *not* interacted with) so the model learns what a "bad" recommendation looks like.

In [5]:
# Split the graph edges for training, validation, and testing
transform = T.RandomLinkSplit(
    num_val=0.1,  # 10% of edges for validation
    num_test=0.1, # 10% of edges for testing
    disjoint_train_ratio=0.3, # Edges used for message passing vs. supervision
    neg_sampling_ratio=1.0,   # 1 negative edge for every 1 positive edge
    add_negative_train_samples=True,
    edge_types=[("user", "rates", "movie")],
    rev_edge_types=[("movie", "rev_rates", "user")],
)

train_data, val_data, test_data = transform(data)

# --- GLOBAL FEATURE INITIALIZATION ---
# Permanently handle missing features for all splits using safe key access
for split in [train_data, val_data, test_data]:
    # Initialize 'user' features if missing (dim: 20)
    if 'x' not in split['user'] or split['user'].x is None:
        split['user'].x = torch.zeros(data['user'].num_nodes, 20)

    # Initialize 'movie' features if missing (dim: 384)
    if 'x' not in split['movie'] or split['movie'].x is None:
        split['movie'].x = torch.zeros(data['movie'].num_nodes, 384)

# Move splits to device
train_data, val_data, test_data = train_data.to(device), val_data.to(device), test_data.to(device)

print(f"Training supervision edges: {train_data['user', 'rates', 'movie'].edge_label_index.shape[1]}")

Training supervision edges: 48402


## 4. GNN Architecture (Encoder & Decoder)
Our architecture has two parts:
1. **The Encoder:** A GNN that passes messages across the graph to generate rich feature embeddings for both Users and Movies. We write it as a standard homogenous model and use PyG's brilliant `to_hetero` function to automatically upgrade it for our complex bipartite graph.
2. **The Decoder:** A simple function that takes a User embedding and a Movie embedding and computes their dot-product. A high dot-product means a high probability of interaction.

In [6]:
class GNNEncoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1, -1), hidden_channels)
        self.conv2 = SAGEConv((-1, -1), out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

class EdgeDecoder(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, user_embeddings, movie_embeddings, edge_index):
        row, col = edge_index
        return (user_embeddings[row] * movie_embeddings[col]).sum(dim=-1)

class RecommenderSystem(torch.nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        self.encoder = GNNEncoder(hidden_channels, hidden_channels)
        self.encoder = to_hetero(self.encoder, data.metadata(), aggr='sum')
        self.decoder = EdgeDecoder()

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        node_embeddings = self.encoder(x_dict, edge_index_dict)
        return self.decoder(
            node_embeddings['user'],
            node_embeddings['movie'],
            edge_label_index
        )

model = RecommenderSystem(hidden_channels=32).to(device)

# Dummy forward pass to trigger lazy initialization
with torch.no_grad():
    model(train_data.x_dict, train_data.edge_index_dict, train_data['user', 'rates', 'movie'].edge_label_index)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 5. Training and Evaluation Loops
We use standard Binary Cross Entropy Loss (`BCEWithLogitsLoss`).
If the edge is a real interaction, the target label is 1. If it's a negative sample, the label is 0.
We evaluate the model using AUC (Area Under the Receiver Operating Characteristic Curve), a standard metric for recommender systems.

In [7]:
def train():
    model.train()
    optimizer.zero_grad()

    edge_label_index = train_data['user', 'rates', 'movie'].edge_label_index
    edge_label = (train_data['user', 'rates', 'movie'].edge_label > 0).float()

    # Cleaner call using the pre-initialized x_dict
    pred = model(train_data.x_dict, train_data.edge_index_dict, edge_label_index)

    loss = F.binary_cross_entropy_with_logits(pred, edge_label)
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(eval_data):
    model.eval()
    edge_label_index = eval_data['user', 'rates', 'movie'].edge_label_index
    edge_label = (eval_data['user', 'rates', 'movie'].edge_label > 0).float()

    pred = model(eval_data.x_dict, eval_data.edge_index_dict, edge_label_index)

    pred_prob = pred.sigmoid().cpu().numpy()
    target = edge_label.cpu().numpy()
    return roc_auc_score(target, pred_prob)

# Execute the training loop
print("Starting Training...")
for epoch in range(1, 201):
    loss = train()
    val_auc = test(val_data)

    if epoch % 50 == 0 or epoch == 1:
        print(f'Epoch: {epoch:02d} | Train Loss: {loss:.4f} | Validation AUC: {val_auc:.4f}')

test_auc = test(test_data)
print(f'\nFinal Test AUC (Recommendation Performance): {test_auc:.4f}')

Starting Training...
Epoch: 01 | Train Loss: 0.6937 | Validation AUC: 0.7206
Epoch: 50 | Train Loss: 0.4609 | Validation AUC: 0.8499
Epoch: 100 | Train Loss: 0.3881 | Validation AUC: 0.8915
Epoch: 150 | Train Loss: 0.3342 | Validation AUC: 0.9134
Epoch: 200 | Train Loss: 0.2972 | Validation AUC: 0.9224

Final Test AUC (Recommendation Performance): 0.9240


## 6. Generating Actual Recommendations
To deploy this, we don't just calculate loss. We take a target user, compute their embedding, calculate the dot product against **all** movie embeddings, and return the movies with the highest scores.

In [8]:
model.eval()
with torch.no_grad():
    # Generate final embeddings using pre-initialized features
    node_embeddings = model.encoder(test_data.x_dict, test_data.edge_index_dict)

    user_id = 0
    user_emb = node_embeddings['user'][user_id]

    movie_embs = node_embeddings['movie']
    scores = (user_emb * movie_embs).sum(dim=-1).sigmoid()

    top_5_indices = scores.argsort(descending=True)[:5]

    print(f"Top 5 Movie Recommendations for User {user_id}:")
    print("-" * 40)
    for i, idx in enumerate(top_5_indices):
        print(f"Rank {i+1}: Movie ID {idx.item()} (Match Score: {scores[idx]:.4f})")

Top 5 Movie Recommendations for User 0:
----------------------------------------
Rank 1: Movie ID 275 (Match Score: 0.9963)
Rank 2: Movie ID 277 (Match Score: 0.9920)
Rank 3: Movie ID 97 (Match Score: 0.9907)
Rank 4: Movie ID 337 (Match Score: 0.9906)
Rank 5: Movie ID 123 (Match Score: 0.9904)
